# Lab 1 — Deploy Parakeet CTC 0.6B with NVIDIA Speech NIM

**Outcome:** deploy the supported Parakeet CTC 0.6B English NIM, verify readiness, transcribe through its HTTP API, and measure client-observed latency and real-time factor. Estimated time: 75–90 minutes; the first NIM startup can spend up to 30 minutes downloading and optimizing artifacts.

This lab follows NVIDIA's Speech NIM deployment workflow using the 0.6B container and its low-memory, single-client offline profile. It requires NVIDIA AI Enterprise entitlement and an NGC API key with Catalog access. Never paste the key into a notebook cell, source file, or saved output.

In [ ]:
from getpass import getpass
from pathlib import Path
import os, statistics, subprocess, time

import requests
import soundfile as sf
from IPython.display import Audio, display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'scripts' / 'start_nim.sh').exists():
    ROOT = ROOT.parent
if not (ROOT / 'scripts' / 'start_nim.sh').exists():
    raise RuntimeError('Open this notebook from the workshop repository.')

CONTAINER_ID = 'parakeet-0-6b-ctc-en-us'
READY_URL = 'http://localhost:9000/v1/health/ready'
TRANSCRIBE_URL = 'http://localhost:9000/v1/audio/transcriptions'

## 1. Start the supported 0.6B NIM

The selected profile is `name=parakeet-0-6b-ctc-en-us,bs=1,mode=ofl,diarizer=disabled,vad=default`. NVIDIA lists it as the single-session offline profile at about 3.08 GB of GPU memory. The helper follows the documented NGC login and Docker flags, mounts a persistent model cache, starts the service on HTTP port 9000 and gRPC port 50051, and waits for `/v1/health/ready`.

In [ ]:
ngc_api_key = getpass('NGC API key (input is hidden): ')
if not ngc_api_key:
    raise RuntimeError('An NGC API key is required for the Speech NIM deployment lab.')

nim_env = os.environ.copy()
nim_env['NGC_API_KEY'] = ngc_api_key
try:
    subprocess.run(['bash', str(ROOT / 'scripts' / 'start_nim.sh')], env=nim_env, check=True)
finally:
    del nim_env['NGC_API_KEY']
    del ngc_api_key

In [ ]:
health = requests.get(READY_URL, timeout=10)
health.raise_for_status()
health.json()

## 2. Copy and inspect the bundled sample

NVIDIA includes an `en-US_sample.wav` file in the NIM container. The HTTP endpoint accepts mono 16-bit WAV, OPUS, or FLAC; keeping this input contract explicit makes client and service behavior easier to debug.

In [ ]:
audio_path = ROOT / 'artifacts' / 'nim' / 'en-US_sample.wav'
audio_path.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(
    ['docker', 'cp', f'{CONTAINER_ID}:/opt/riva/wav/en-US_sample.wav', str(audio_path)],
    check=True,
)
audio_info = sf.info(str(audio_path))
audio_seconds = audio_info.frames / audio_info.samplerate
print({'path': str(audio_path), 'sample_rate': audio_info.samplerate, 'duration_seconds': round(audio_seconds, 2)})
display(Audio(filename=str(audio_path)))

## 3. Transcribe through the NIM HTTP API

Offline transcription sends the complete file in one multipart request to `/v1/audio/transcriptions`. This is the production service boundary: the client supplies audio and language, while the NIM owns preprocessing, optimized inference, and decoding.

In [ ]:
def transcribe_file(path: Path) -> dict:
    with path.open('rb') as audio_file:
        response = requests.post(
            TRANSCRIBE_URL,
            data={'language': 'en-US'},
            files={'file': (path.name, audio_file, 'audio/wav')},
            timeout=600,
        )
    response.raise_for_status()
    return response.json()

prediction = transcribe_file(audio_path)
prediction

## 4. Benchmark the service boundary

RTF is end-to-end request latency divided by audio duration; lower is better. This includes local HTTP and service overhead but not a real network hop. A production capacity test still needs representative audio lengths, concurrency, arrival patterns, warm/cold behavior, percentiles, errors, and an explicit accuracy set.

In [ ]:
transcribe_file(audio_path)  # warm-up
latencies = []
for _ in range(5):
    started = time.perf_counter()
    transcribe_file(audio_path)
    latencies.append(time.perf_counter() - started)

median_latency = statistics.median(latencies)
metrics = {
    'audio_seconds': audio_seconds,
    'median_latency_seconds': median_latency,
    'real_time_factor': median_latency / audio_seconds,
    'throughput_x_realtime': audio_seconds / median_latency,
    'requests': len(latencies),
}
metrics

## Production mapping and cleanup

The NIM container packages the supported TensorRT/Triton serving stack. For production, move the same container/profile and API contract behind Kubernetes or EKS with an NGC pull secret, persistent model cache, readiness/liveness checks, internal services, TLS, metrics, GPU scheduling, autoscaling, request limits, and load testing. NVIDIA AI Enterprise licensing remains a deployment prerequisite.

**Checkpoint:** record the returned transcript, readiness response, selected profile, median latency, and RTF. Then stop the NIM before Lab 2 so it releases GPU memory:

```bash
bash scripts/stop_nim.sh
```